In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)

import lightgbm as lgb

from src.data import aggregate_bureau, aggregate_previous
from src.features import build_features

RANDOM_STATE = 42

print("Imports successful.")
print("LightGBM:", lgb.__version__)

Imports successful.
LightGBM: 4.7.0


In [2]:
previous = pd.read_csv("../data/raw/previous_application.csv")

print("Shape:", previous.shape)
print("Unique applicants:", previous["SK_ID_CURR"].nunique())

print("\nApplications per applicant:")
print(previous.groupby("SK_ID_CURR").size().describe())

Shape: (1670214, 37)
Unique applicants: 338857

Applications per applicant:
count    338857.000000
mean          4.928964
std           4.220716
min           1.000000
25%           2.000000
50%           4.000000
75%           7.000000
max          77.000000
dtype: float64


In [3]:
print("Columns:", previous.shape[1])
print("\nFirst 5 rows:")
display(previous.head())

print("\nContract status:")
print(previous["NAME_CONTRACT_STATUS"].value_counts())

print("\nContract type:")
print(previous["NAME_CONTRACT_TYPE"].value_counts())

print("\nYield group:")
print(previous["NAME_YIELD_GROUP"].value_counts())

print("\nTop missing-value rates:")
print(
    previous.isna()
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

Columns: 37

First 5 rows:


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN



Contract status:
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

Contract type:
NAME_CONTRACT_TYPE
Cash loans         747553
Consumer loans     729151
Revolving loans    193164
XNA                   346
Name: count, dtype: int64

Yield group:
NAME_YIELD_GROUP
XNA           517215
middle        385532
high          353331
low_normal    322095
low_action     92041
Name: count, dtype: int64

Top missing-value rates:
RATE_INTEREST_PRIVILEGED     0.996437
RATE_INTEREST_PRIMARY        0.996437
AMT_DOWN_PAYMENT             0.536365
RATE_DOWN_PAYMENT            0.536365
NAME_TYPE_SUITE              0.491198
DAYS_TERMINATION             0.402981
DAYS_FIRST_DRAWING           0.402981
DAYS_FIRST_DUE               0.402981
DAYS_LAST_DUE_1ST_VERSION    0.402981
DAYS_LAST_DUE                0.402981
dtype: float64


In [4]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("CV splitter ready.")

CV splitter ready.


In [5]:
app = pd.read_csv("../data/raw/application_train.csv")

bureau = pd.read_csv("../data/raw/bureau.csv")
buro_agg = aggregate_bureau(bureau)
del bureau

previous = pd.read_csv("../data/raw/previous_application.csv")
prev_agg = aggregate_previous(previous)
del previous

app_full = (
    app
    .merge(buro_agg, on="SK_ID_CURR", how="left")
    .merge(prev_agg, on="SK_ID_CURR", how="left")
)

print("Shape after both joins:", app_full.shape)

Shape after both joins: (307511, 157)


In [6]:
X_full = app_full.drop(columns=["TARGET", "SK_ID_CURR"])
y_full = app_full["TARGET"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print("Development set:", X_dev.shape)
print("Test set:", X_test.shape)
print("Development default rate:", round(y_dev.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))

Development set: (246008, 155)
Test set: (61503, 155)
Development default rate: 0.0807
Test default rate: 0.0807


In [7]:
X_dev = build_features(X_dev)

print("Shape:", X_dev.shape)
print(
    "Infinite values:",
    np.isinf(X_dev.select_dtypes("number")).sum().sum()
)

Shape: (246008, 164)
Infinite values: 0


In [8]:
model = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    verbose=-1,
    n_jobs=2,
)

print("Model ready.")

Model ready.


In [9]:
aucs = cross_val_score(
    model,
    X_dev,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"Full feature set: {aucs.mean():.4f} ± {aucs.std():.4f}")

Full feature set: 0.7686 ± 0.0017


In [11]:
pd.DataFrame([
    {"features": "application only",       "cv_auc": 0.7536, "cv_std": 0.0019},
    {"features": "+ ratios",               "cv_auc": 0.7601, "cv_std": 0.0009},
    {"features": "+ bureau",               "cv_auc": 0.7647, "cv_std": 0.0011},
    {"features": "+ previous application", "cv_auc": 0.7686, "cv_std": 0.0017},
]).to_csv("../reports/table_contributions.csv", index=False)

## Previous Application Findings

Aggregating 1,670,214 previous applications for 338,857 applicants into 20
summary columns raised CV ROC-AUC from 0.7647 ± 0.0011 to 0.7686 ± 0.0017,
a gain of +0.0039.

The contract status breakdown is the most interesting part of this table:
1,036,781 approved, 316,319 cancelled, 290,678 refused and 26,436 unused
offers. A refusal rate feature captures how often Home Credit itself has
previously turned this applicant down.

That raises a question worth naming. A model that leans on past refusals is
partly learning to agree with the company's own historical decisions rather
than learning about repayment behaviour. If those past decisions were wrong
or biased, the model inherits that. This is a reason to check feature
importance rather than accept the gain uncritically.

### Contribution of each data source

| Features | CV AUC | Gain |
|---|---:|---:|
| Application only | 0.7536 | — |
| + 5 engineered ratios | 0.7601 | +0.0065 |
| + bureau (1.7M rows → 17 cols) | 0.7647 | +0.0046 |
| + previous applications (1.67M rows → 20 cols) | 0.7686 | +0.0039 |

Five hand-built ratio features contributed more than either million-row
table. Each additional data source also returned less than the one before.